# Ensemble Learning Techniques: A Walkthrough with Random Forests in Python

Complete local CPU examples. Iris supports the multiclass examples; diabetes supplies a real regression target and breast-cancer data supplies binary targets. Test partitions are held out of fitting and model-selection callbacks. Single fixed splits are demonstrations, not universal performance guarantees. Install dependencies:

```bash
python -m pip install numpy==1.26.4 scipy==1.17.1 scikit-learn==1.9.1 xgboost==3.2.0 lightgbm==4.7.0 joblib==1.6.0 threadpoolctl==3.7.0
```

On macOS, the XGBoost/LightGBM wheels also need an available OpenMP runtime (libomp).

In [ ]:
import sys, json
from importlib.metadata import version
import numpy as np
from sklearn.datasets import load_iris, load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import (BaggingClassifier, BaggingRegressor, RandomForestClassifier,
                              GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import accuracy_score, r2_score, mean_absolute_error
print('Python:', sys.version.split()[0])
print('scikit-learn:', version('scikit-learn'), 'XGBoost:', version('xgboost'), 'LightGBM:', version('lightgbm'))

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)
print('Iris train/test:', X_train.shape, X_test.shape)
print('Train class counts:', np.bincount(y_train).tolist())

In [ ]:
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=10, max_samples=0.5, max_features=0.5, random_state=42, n_jobs=1)
bagging.fit(X_train, y_train)
bagging_prediction = bagging.predict(X_test)
print('Bagging accuracy:', f'{accuracy_score(y_test, bagging_prediction):.6f}')
print('Per-estimator training rows/features:', len(bagging.estimators_samples_[0]), len(bagging.estimators_features_[0]))

In [ ]:
diabetes = load_diabetes()
R_train, R_test, r_train, r_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42)
bagging_regressor = BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=10, random_state=42, n_jobs=1)
bagging_regressor.fit(R_train, r_train)
regression_prediction = bagging_regressor.predict(R_test)
print('Diabetes train/test:', R_train.shape, R_test.shape)
print('Bagging regression R2:', f'{r2_score(r_test, regression_prediction):.6f}')
print('Mean absolute error:', f'{mean_absolute_error(r_test, regression_prediction):.6f}')

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
forest.fit(X_train, y_train)
forest_probabilities = forest.predict_proba(X_test)
print('Random forest accuracy:', f'{forest.score(X_test, y_test):.6f}')
print('Number of trees:', len(forest.estimators_))

In [ ]:
base_classifiers = [RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
                    GradientBoostingClassifier(n_estimators=100, random_state=42)]
oof = np.zeros((len(X_train), len(base_classifiers)))
seen = np.zeros(len(X_train), dtype=int)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_membership = []
for train_index, val_index in folds.split(X_train, y_train):
    assert not set(train_index) & set(val_index)
    seen[val_index] += 1
    fold_membership.append((train_index.copy(), val_index.copy()))
    for i, estimator in enumerate(base_classifiers):
        fold_model = clone(estimator).fit(X_train[train_index], y_train[train_index])
        oof[val_index, i] = fold_model.predict(X_train[val_index])
assert np.all(seen == 1)
meta_classifier = LogisticRegression(max_iter=1000).fit(oof, y_train)
# Refit new base models on ALL training rows before predicting held-out test rows.
fitted_base = [clone(estimator).fit(X_train, y_train) for estimator in base_classifiers]
test_meta_features = np.column_stack([estimator.predict(X_test) for estimator in fitted_base])
stacked_prediction = meta_classifier.predict(test_meta_features)
print('OOF feature matrix:', oof.shape, 'Test feature matrix:', test_meta_features.shape)
print('Validation predictions per training row:', np.unique(seen).tolist())
print('Stacked ensemble accuracy:', f'{accuracy_score(y_test, stacked_prediction):.6f}')

In [ ]:
adaboost = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost.fit(X_train, y_train)
print('AdaBoost accuracy:', f'{adaboost.score(X_test, y_test):.6f}')

In [ ]:
binary = load_breast_cancer()
B_train_full, B_test, b_train_full, b_test = train_test_split(
    binary.data, binary.target, test_size=0.2, random_state=42, stratify=binary.target)
B_train, B_valid, b_train, b_valid = train_test_split(
    B_train_full, b_train_full, test_size=0.2, random_state=42, stratify=b_train_full)
print('Binary train/validation/test:', B_train.shape, B_valid.shape, B_test.shape)
print('Class labels:', np.unique(binary.target).tolist())

In [ ]:
import xgboost as xgb
xgboost = xgb.XGBClassifier(objective='binary:logistic', colsample_bytree=0.3,
    learning_rate=0.1, max_depth=5, reg_alpha=10, n_estimators=100, random_state=42, n_jobs=1)
xgboost.fit(B_train, b_train)
xgb_probabilities = xgboost.predict_proba(B_test)[:, 1]
print('XGBoost test accuracy:', f'{accuracy_score(b_test, xgboost.predict(B_test)):.6f}')

In [ ]:
import lightgbm as lgb
lgb_train = lgb.Dataset(B_train, label=b_train)
lgb_valid = lgb.Dataset(B_valid, label=b_valid, reference=lgb_train)
params = {'boosting_type': 'gbdt', 'objective': 'binary', 'metric': 'binary_logloss',
          'num_leaves': 8, 'min_data_in_leaf': 5, 'learning_rate': 0.1,
          'feature_fraction': 0.9, 'seed': 42, 'num_threads': 1,
          'deterministic': True, 'force_col_wise': True}
gbm = lgb.train(params, lgb_train, num_boost_round=200,
    valid_sets=[lgb_train, lgb_valid], valid_names=['train', 'valid'],
    callbacks=[lgb.log_evaluation(period=50)])
lgb_probabilities = gbm.predict(B_test)
print('LightGBM test accuracy:', f'{accuracy_score(b_test, (lgb_probabilities >= 0.5).astype(int)):.6f}')

In [ ]:
# Independent aggregation and metric checks, not just successful .fit calls.
bagging_mean = np.mean([tree.predict_proba(X_test[:, features])
    for tree, features in zip(bagging.estimators_, bagging.estimators_features_)], axis=0)
np.testing.assert_allclose(bagging.predict_proba(X_test), bagging_mean, atol=1e-12)
forest_mean = np.mean([tree.predict_proba(X_test) for tree in forest.estimators_], axis=0)
np.testing.assert_allclose(forest_probabilities, forest_mean, atol=1e-12)
np.testing.assert_allclose(bagging_regressor.score(R_test, r_test), r2_score(r_test, regression_prediction))
np.testing.assert_array_equal(meta_classifier.predict(test_meta_features), stacked_prediction)
assert len(fold_membership) == 5 and np.all(seen == 1)
assert all(len(model.estimators_) == 100 for model in [forest, fitted_base[0]])
assert set(np.unique(binary.target)) == {0, 1}
for probabilities in [xgb_probabilities, lgb_probabilities]:
    assert probabilities.shape == b_test.shape and np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
np.testing.assert_array_equal((xgb_probabilities > 0.5).astype(int), xgboost.predict(B_test))
assert np.isfinite(regression_prediction).all()
assert all(len(set(train) & set(valid)) == 0 for train, valid in fold_membership)
print('PASS: independent probability aggregation, regression metrics, OOF coverage/disjoint folds, full refits and finite binary predictions.')

## Original source-code appendix

These seven original blocks are retained verbatim as nonexecuting historical source. Use the maintained examples above.

```python
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
bagging = BaggingClassifier(base_estimator=DecisionTreeClassifier(),n_estimators=10, max_samples=0.5, max_features=0.5)
```

```python
bagging.fit(X_train, y_train)
bagging.score(X_test,y_test)
```

```python
from sklearn.ensemble import BaggingRegressor
bagging = BaggingRegressor(DecisionTreeRegressor())
bagging.fit(X_train, y_train)
model.score(X_test,y_test)
```

```python
# Load the dataset
data = load_iris()
X, y = data.data, data.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define base classifiers
base_classifiers = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    GradientBoostingClassifier(n_estimators=100, random_state=42)
]

# Define a meta-classifier
meta_classifier = LogisticRegression()

# Create an array to hold the predictions from base classifiers
base_classifier_predictions = np.zeros((len(X_train), len(base_classifiers)))

# Perform stacking using K-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_index, val_index in kf.split(X_train):
    train_fold, val_fold = X_train[train_index], X_train[val_index]
    train_target, val_target = y_train[train_index], y_train[val_index]

    for i, clf in enumerate(base_classifiers):
        cloned_clf = clone(clf)
        cloned_clf.fit(train_fold, train_target)
        base_classifier_predictions[val_index, i] = cloned_clf.predict(val_fold)

# Train the meta-classifier on base classifier predictions
meta_classifier.fit(base_classifier_predictions, y_train)

# Make predictions using the stacked ensemble
stacked_predictions = np.zeros((len(X_test), len(base_classifiers)))
for i, clf in enumerate(base_classifiers):
    stacked_predictions[:, i] = clf.predict(X_test)

# Make final predictions using the meta-classifier
final_predictions = meta_classifier.predict(stacked_predictions)

# Evaluate the stacked ensemble's performance
accuracy = accuracy_score(y_test, final_predictions)
print(f"Stacked Ensemble Accuracy: {accuracy:.2f}")
```

```python
from sklearn.ensemble import AdaBoostClassifier
model = AdaBoostClassifier(n_estimators=100)
model.fit(X_train, y_train)
model.score(X_test,y_test)
```

```python
import xgboost as xgb
params = {"objective":"binary:logistic",'colsample_bytree': 0.3,'learning_rate': 0.1,
                'max_depth': 5, 'alpha': 10}
model = xgb.XGBClassifier(**params)
model.fit(X_train, y_train)
model.fit(X_train, y_train)
model.score(X_test,y_test)
```

```python
import lightgbm as lgb
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train)
params = {'boosting_type': 'gbdt',
              'objective': 'binary',
              'num_leaves': 40,
              'learning_rate': 0.1,
              'feature_fraction': 0.9
              }
gbm = lgb.train(params,
    lgb_train,
    num_boost_round=200,
    valid_sets=[lgb_train, lgb_eval],
    valid_names=['train','valid'],
   )
```